In [1]:
# collect_and_raster_carla.py  — balanced micro-episodes (6 s) for diverse trajectories
# (persistent balancing + lane-change detection + AGENTS image pruner + soft targets + mild turn & lane boosts)
import argparse, time, os, shutil, multiprocessing, datetime, queue, traceback, random, json, math, glob
from threading import Event, Thread
from typing import Optional, List, Tuple, Dict

import numpy as np
from PIL import Image, ImageDraw

# ==== Project imports (keep your path) ====
import sys
sys.path.append('./UTDAutopilot')

from AutopilotMain.autopilot.environment.carla import CarlaEnvironment
from AutopilotMain.autopilot.environment.carla import CarlaRecorder
from AutopilotMain.autopilot.agent import AutonomousVehicle

# -------------------------- headless-safe (docker/X11)
os.environ["SDL_VIDEODRIVER"] = "offscreen"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-carla"
os.makedirs("/tmp/runtime-carla", exist_ok=True)

WEATHER_PRESETS = [
    "ClearNoon","CloudyNoon","WetNoon","MidRainyNoon","SoftRainNoon",
    "ClearSunset","CloudySunset","WetSunset","WetCloudySunset","SoftRainSunset",
]

# =================== map raster helpers ===================
import xml.etree.ElementTree as ET

def _rot2d(th):
    c,s=math.cos(th),math.sin(th)
    return np.array([[c,-s],[s,c]],np.float32)

def world_to_ego_pixels(xy_world, ego_xy, ego_yaw_deg, ppm, img_size):
    xy = xy_world - np.array(ego_xy, np.float32)[None,:]
    xy = xy @ _rot2d(-math.radians(ego_yaw_deg)).T
    xy[:,1] *= -1.0
    xy = xy * ppm
    c = img_size//2
    xy[:,0] += c; xy[:,1] += c
    return xy

def build_static_lane_polylines(world, sample_res_m=1.5):
    carla_map = world.get_map()
    topology = carla_map.get_topology()
    from collections import defaultdict
    chains = defaultdict(list)
    for wp_start, _ in topology:
        key=(wp_start.road_id, wp_start.section_id, wp_start.lane_id)
        wps=[wp_start]; curr=wp_start
        seen={(curr.road_id, curr.section_id, curr.lane_id, int(curr.s))}
        while True:
            nxt=curr.next(sample_res_m)
            if not nxt: break
            curr=nxt[0]
            tag=(curr.road_id, curr.section_id, curr.lane_id, int(curr.s))
            if tag in seen: break
            seen.add(tag)
            if curr.lane_id!=wp_start.lane_id or curr.road_id!=wp_start.road_id: break
            wps.append(curr)
        if len(wps)>=2: chains[key].append(wps)

    def wps_xy(wps):
        return np.array([[wp.transform.location.x, wp.transform.location.y] for wp in wps], np.float32)

    center, left_b, right_b, drivable, center_meta = [], [], [], [], []
    for key, seqs in chains.items():
        seqs = sorted(seqs, key=lambda s: s[0].s)
        merged=[]; last=None
        for s in seqs:
            if not merged: merged.extend(s)
            else:
                if last and s[0].road_id==last.road_id and s[0].lane_id==last.lane_id:
                    merged.extend(s)
            last=s[-1]
        if len(merged)<2: continue
        xy=wps_xy(merged)
        center.append(xy)
        center_meta.append({"road_id": int(key[0]), "section_id": int(key[1]), "lane_id": int(key[2]), "poly": [[float(a), float(b)] for a,b in xy]})
        L,R=[],[]
        for wp in merged:
            loc=wp.transform.location
            yaw=math.radians(wp.transform.rotation.yaw)
            nx,ny=-math.sin(yaw), math.cos(yaw)
            w=wp.lane_width*0.5
            L.append([loc.x+nx*w, loc.y+ny*w])
            R.append([loc.x-nx*w, loc.y-ny*w])
        L=np.array(L,np.float32); R=np.array(R,np.float32)
        left_b.append(L); right_b.append(R)
        for i in range(len(L)-1):
            quad=np.stack([L[i], L[i+1], R[i+1], R[i]], 0)
            drivable.append(quad)

    return {"center":center, "left_boundary":left_b, "right_boundary":right_b,
            "drivable":drivable, "center_meta":center_meta}

def parse_opendrive_crosswalks_and_stoplines(xodr_text):
    crosswalks, stop_lines = [], []
    try:
        root = ET.fromstring(xodr_text)
    except Exception:
        return crosswalks, stop_lines
    for road in root.findall(".//road"):
        rid = road.get("id","")
        for obj in road.findall(".//objects/object"):
            typ = (obj.get("type","")+":"+obj.get("subtype","")).lower()
            if "crosswalk" in typ or "crossing" in typ:
                s = float(obj.get("s","0")); t=float(obj.get("t","0"))
                length=float(obj.get("length","3.0")); width=float(obj.get("width","6.0"))
                half_s=max(1.5, length*0.5); half_t=max(1.5, width*0.5)
                poly_st=[(s-half_s,t-half_t),(s+half_s,t-half_t),(s+half_s,t+half_t),(s-half_s,t+half_t)]
                crosswalks.append({"road_id":rid, "poly_st":poly_st})
    return crosswalks, stop_lines

def project_st_to_world(world, road_id, s, t):
    cm=world.get_map()
    cands=[w for w in cm.generate_waypoints(2.0) if str(w.road_id)==str(road_id)]
    if not cands: return None
    w=min(cands, key=lambda wp: abs(wp.s - s))
    loc=w.transform.location; yaw=math.radians(w.transform.rotation.yaw)
    nx,ny=-math.sin(yaw), math.cos(yaw)
    return (loc.x + nx*t, loc.y + ny*t)

def crosswalk_polys_world(world, xodr_text):
    cw,_ = parse_opendrive_crosswalks_and_stoplines(xodr_text)
    polys=[]
    for z in cw:
        rid=z["road_id"]
        poly=[]
        for (s,t) in z["poly_st"]:
            p=project_st_to_world(world, rid, s,t)
            if p is not None: poly.append((float(p[0]), float(p[1])))
        if len(poly)>=3: polys.append(poly)
    return polys

class OpenDriveRasterizer:
    def __init__(self, layers, img_size=512, meters_ahead=50.0, meters_behind=20.0, meters_side=35.0):
        self.layers=layers; self.img_size=img_size
        m_per_px=max((meters_ahead+meters_behind)/img_size, (2*meters_side)/img_size)
        self.ppm=1.0/m_per_px
        self.meters_ahead=meters_ahead; self.meters_behind=meters_behind; self.meters_side=meters_side
        self.L_BG, self.L_DRIV, self.L_BOUND, self.L_CENT = 0,1,2,3

    def _clip(self, pts):
        H=W=self.img_size
        m=(pts[:,0]>=-5)&(pts[:,0]<=W+5)&(pts[:,1]>=-5)&(pts[:,1]<=H+5)
        return pts[m]

    def _fill_poly_to_mask(self, mask, pts, val):
        im=Image.new("L",(self.img_size,self.img_size),0)
        ImageDraw.Draw(im).polygon([tuple(p) for p in pts.tolist()], fill=val)
        arr=np.array(im,np.uint8); np.maximum(mask, arr, out=mask)

    def _line_to_mask(self, mask, pts, w, val):
        im=Image.new("L",(self.img_size,self.img_size),0)
        ImageDraw.Draw(im).line([tuple(p) for p in pts.tolist()], width=w, fill=val)
        arr=np.array(im,np.uint8); np.maximum(mask, arr, out=mask)

    def render(self, ego_xy, ego_yaw_deg):
        H=W=self.img_size
        rgba=Image.new("RGBA",(W,H),(0,0,0,255)); d=ImageDraw.Draw(rgba,"RGBA")
        mask=np.zeros((H,W),np.uint8)

        for quad in self.layers["drivable"]:
            pts=world_to_ego_pixels(quad.copy(), ego_xy, ego_yaw_deg, self.ppm, self.img_size)
            if len(pts)==4:
                d.polygon([tuple(p) for p in pts.tolist()], fill=(230,230,230,255))
                self._fill_poly_to_mask(mask, pts, self.L_DRIV)

        for poly in (self.layers["left_boundary"]+self.layers["right_boundary"]):
            pts=world_to_ego_pixels(poly.copy(), ego_xy, ego_yaw_deg, self.ppm, self.img_size)
            pts=self._clip(pts)
            if len(pts)>=2:
                d.line([tuple(p) for p in pts.tolist()], width=2, fill=(130,130,130,255))
                self._line_to_mask(mask, pts, 2, self.L_BOUND)

        for poly in self.layers["center"]:
            pts=world_to_ego_pixels(poly.copy(), ego_xy, ego_yaw_deg, self.ppm, self.img_size)
            pts=self._clip(pts)
            if len(pts)>=2:
                d.line([tuple(p) for p in pts.tolist()], width=1, fill=(0,200,255,255))
                self._line_to_mask(mask, pts, 1, self.L_CENT)

        return rgba, Image.fromarray(mask, "L")

    def get_view_params(self):
        return self.ppm, self.img_size, self.meters_ahead, self.meters_behind, self.meters_side

# ================= heartbeat & sync =================

def start_heartbeat(mq, stop_event, interval=5.0):
    def _beat():
        while not stop_event.is_set():
            try: mq.put(('hb', time.time()))
            except: pass
            time.sleep(interval)
    t=Thread(target=_beat, daemon=True); t.start(); return t

def configure_sync(env, fps=10, carla_port=2000, tm_port=None, seed=0):
    import carla
    world = env.world
    client = getattr(env, "client", None)
    if client is None:
        client = carla.Client('localhost', carla_port); client.set_timeout(30.0)

    tm = client.get_trafficmanager(tm_port) if tm_port else client.get_trafficmanager()
    try: tm.set_random_device_seed(int(seed))
    except Exception: pass

    orig_settings = world.get_settings()
    new_settings = world.get_settings()
    new_settings.synchronous_mode = True
    new_settings.fixed_delta_seconds = 1.0 / float(fps)
    world.apply_settings(new_settings)

    try: tm.set_synchronous_mode(True)
    except Exception: pass
    try: tm.set_hybrid_physics_mode(True)
    except Exception: pass
    try:
        tm.global_distance_to_leading_vehicle(2.5)
        tm.global_percentage_speed_difference(-10)
    except Exception: pass

    for _ in range(10):
        world.tick()
    return orig_settings, tm

# === Kinematic sanity helpers ===
MAX_STEP_M_0P1S = 6.0
MAX_YAW_DEG_0P1S = 120.0

def _wrap180(a_deg: float) -> float:
    return ((a_deg + 180.0) % 360.0) - 180.0

def _ang_diff_deg(a: float, b: float) -> float:
    return abs(_wrap180(a - b))

def speed_mag(v): return float((v[0]**2+v[1]**2+v[2]**2)**0.5)

# ================= Balancing bins =================
BIN_KEYS = [
    "STOP",
    "STR_P0_0to8", "STR_P1_8to18", "STR_P2_18to30", "STR_P3_30plus",
    "LTURN_MILD", "LTURN_MED", "LTURN_HARD",
    "RTURN_MILD", "RTURN_MED", "RTURN_HARD",
    "LANE_LEFT", "LANE_RIGHT",
]
MOVEMENT_KEYS = [k for k in BIN_KEYS if k != "STOP"]

# ---- STOP thresholds (tightened, tunable)
OLD_STOP_LEN = 3.0
OLD_STOP_YF  = 2.0
STOP_LEN_TIGHT = 1.8  # default; CLI overrides
STOP_YF_TIGHT  = 1.4  # default; CLI overrides

def classify_future_2hz(fut_xy_ego_yfwd_2hz: np.ndarray) -> str:
    """
    Classify using *tight* STOP rule (both short path & tiny forward progress).
    """
    xy = np.asarray(fut_xy_ego_yfwd_2hz, np.float32)
    y_final = float(xy[-1,1])
    length = float(np.linalg.norm(np.diff(xy, axis=0), axis=1).sum())
    v_last = xy[-1] - xy[-2] if len(xy) >= 2 else np.array([0., 1.], np.float32)
    theta_deg = math.degrees(math.atan2(v_last[0], v_last[1]))  # vs +Y

    if (length < float(STOP_LEN_TIGHT)) and (y_final < float(STOP_YF_TIGHT)):
        return "STOP"

    if abs(theta_deg) < 12.0:
        if y_final < 8.0:   return "STR_P0_0to8"
        if y_final < 18.0:  return "STR_P1_8to18"
        if y_final < 30.0:  return "STR_P2_18to30"
        return "STR_P3_30plus"

    side = "L" if theta_deg > 0 else "R"
    mag = abs(theta_deg)
    if mag < 25.0: level = "MILD"
    elif mag < 45.0: level = "MED"
    else: level = "HARD"
    return f"{'LTURN' if side=='L' else 'RTURN'}_{level}"

# ============ persistent balance helpers ============
def _load_json(path, default):
    try:
        with open(path, "r") as f: return json.load(f)
    except Exception:
        return default

def _save_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2)

def _zeros_dict():
    return {k: 0 for k in BIN_KEYS}

# ================= Episode planning helpers (RESTORED/EXTENDED) =================
def find_waypoints(world, step=10.0):
    m = world.get_map()
    return m.generate_waypoints(step)

def pick_straight_wp(wps):
    random.shuffle(wps)
    for wp in wps:
        nxt = wp.next(20.0)
        if not nxt: continue
        yaw0 = wp.transform.rotation.yaw
        yaw1 = nxt[0].transform.rotation.yaw
        if _ang_diff_deg(yaw0, yaw1) < 5.0:
            return wp
    return random.choice(wps) if wps else None

def pick_junction_approach_wp(wps):
    random.shuffle(wps)
    for wp in wps:
        if wp.is_junction: continue
        nxt = wp.next(18.0)
        if not nxt: continue
        if nxt[0].is_junction:
            return wp
    return random.choice(wps) if wps else None

def pick_gentle_curve_wp(wps, min_deg=10.0, max_deg=22.0, lookahead=22.0):
    """
    Try to find a segment that gently bends (good for MILD). Falls back random.
    """
    cand=[]
    for wp in random.sample(wps, min(len(wps), 400)):
        nxt = wp.next(lookahead)
        if not nxt: continue
        yaw0 = wp.transform.rotation.yaw
        yaw1 = nxt[0].transform.rotation.yaw
        d = _ang_diff_deg(yaw0, yaw1)
        if min_deg <= d <= max_deg:
            cand.append(wp)
    if cand: return random.choice(cand)
    return random.choice(wps) if wps else None

# >>> lane-adj helper (cross-version safe)
def _has_adjacent_lane(wp, side="left"):
    method_names = ("get_left_lane", "get_left") if side == "left" else ("get_right_lane", "get_right")
    for name in method_names:
        fn = getattr(wp, name, None)
        if callable(fn):
            try:
                adj = fn()
            except TypeError:
                try:
                    adj = fn(wp)
                except Exception:
                    adj = None
            except Exception:
                adj = None
            if adj is not None:
                return True
    try:
        import carla
        lc = getattr(wp, "lane_change", None)
        if lc is not None:
            if side == "left"  and int(lc) & int(carla.LaneChange.Left):  return True
            if side == "right" and int(lc) & int(carla.LaneChange.Right): return True
    except Exception:
        pass
    return False

def pick_multilane_wp(wps, side="left"):
    if not wps: return None
    wps = list(wps)
    random.shuffle(wps)
    for wp in wps:
        try:
            if _has_adjacent_lane(wp, side=side):
                return wp
        except Exception:
            continue
    return random.choice(wps)

# ================= Per-episode behavior =================
class EpisodeBias:
    def __init__(self, key:str):
        self.key = key

    def apply_tick(self, ego_actor, tm, t_ep: float):
        import carla
        key = self.key

        # STOP safety (outer loop also enforces)
        if key == "STOP":
            try:
                ego_actor.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, steer=0.0, hand_brake=True))
            except Exception: pass
            return

        # speed caps
        slow_cap = None
        if key.startswith("STR_P0"): slow_cap = -80
        elif key.startswith("STR_P1"): slow_cap = -60
        elif key.startswith("STR_P2"): slow_cap = -40
        elif key.startswith("STR_P3"): slow_cap = -20
        elif "_HARD" in key: slow_cap = -50
        elif "_MED" in key:  slow_cap = -40
        elif "_MILD" in key: slow_cap = -30

        if slow_cap is not None:
            try: tm.vehicle_percentage_speed_difference(ego_actor, int(slow_cap))
            except Exception: pass

        # turn nudges (lighter for MILD)
        steer = 0.0
        if key.startswith("LTURN"):
            steer = +0.15 if "MILD" in key else (+0.28 if "MED" in key else +0.45)
        elif key.startswith("RTURN"):
            steer = -0.15 if "MILD" in key else (-0.28 if "MED" in key else -0.45)

        if steer != 0.0 and t_ep < 2.5:
            try:
                ego_actor.apply_control(carla.VehicleControl(steer=float(steer)))
            except Exception:
                pass

        # lane-change: stronger/longer enforcement + nudges
        if key in ("LANE_LEFT","LANE_RIGHT"):
            try:
                # enable auto lane change just in case; then force repeatedly
                tm.auto_lane_change(ego_actor, True)
            except Exception:
                pass
            if t_ep < 4.0:
                try:
                    tm.force_lane_change(ego_actor, True if key=="LANE_LEFT" else False)
                except Exception:
                    pass
                # gentle manual nudge to help TM commit
                nudge = +0.20 if key=="LANE_LEFT" else -0.20
                try:
                    ego_actor.apply_control(carla.VehicleControl(steer=float(nudge)))
                except Exception:
                    pass

# ================ optional lattice (for reporting only) ================
def load_lattice(path: Optional[str]):
    if not path or not os.path.exists(path): return None
    import pickle
    with open(path,"rb") as f:
        L = pickle.load(f)
    L = np.asarray(L, np.float32)  # (M,12,2)
    return L

def _delta(a,b):
    return float(np.linalg.norm(a-b, axis=1).max())

def nearest_mode_idx(L: np.ndarray, fut12: np.ndarray) -> int:
    return int(np.argmin([_delta(fut12, m) for m in L]))

def resample_to_T(xy: np.ndarray, T: int) -> np.ndarray:
    xy = np.asarray(xy, np.float32)
    if xy.shape[0] == T: return xy
    t0 = np.linspace(0.0, 1.0, num=xy.shape[0], dtype=np.float32)
    t1 = np.linspace(0.0, 1.0, num=T,           dtype=np.float32)
    out = np.empty((T, 2), np.float32)
    for j in range(2):
        out[:, j] = np.interp(t1, t0, xy[:, j])
    return out

# ---------- Lane-change classification (relaxed) ----------
def classify_lane_change(world, P_world: np.ndarray, fut12_ego_yfwd: np.ndarray) -> Optional[str]:
    """
    Return 'LANE_LEFT' or 'LANE_RIGHT' if we detect a lane change; else None.
    Prefers CARLA map lane_id change; falls back to lateral-offset heuristic.
    """
    try:
        import carla
    except Exception:
        carla = None

    if fut12_ego_yfwd.shape[0] < 2:
        return None
    y_final = float(fut12_ego_yfwd[-1, 1])
    v_last  = fut12_ego_yfwd[-1] - fut12_ego_yfwd[-2]
    theta   = math.degrees(math.atan2(float(v_last[0]), float(v_last[1])))

    # more permissive than before
    if y_final < 6.0 or abs(theta) >= 10.0:
        return None

    # Map-based lane_id change
    if carla is not None:
        try:
            m = world.get_map()
            p0 = carla.Location(x=float(P_world[0,0]),  y=float(P_world[0,1]),  z=0.0)
            p1 = carla.Location(x=float(P_world[-1,0]), y=float(P_world[-1,1]), z=0.0)
            wp0 = m.get_waypoint(p0, project_to_road=True, lane_type=carla.LaneType.Driving)
            wp1 = m.get_waypoint(p1, project_to_road=True, lane_type=carla.LaneType.Driving)
            if wp0 is not None and wp1 is not None and not (wp0.is_junction or wp1.is_junction):
                if wp0.road_id == wp1.road_id and wp0.section_id == wp1.section_id and wp0.lane_id != wp1.lane_id:
                    return "LANE_LEFT" if int(wp1.lane_id) > int(wp0.lane_id) else "LANE_RIGHT"
                # lateral-offset fallback using lane width (relaxed)
                lane_w = float(getattr(wp0, "lane_width", 3.5))
                x_final = float(fut12_ego_yfwd[-1, 0])
                if abs(x_final) > 0.5 * lane_w:
                    return "LANE_LEFT" if x_final > 0 else "LANE_RIGHT"
        except Exception:
            pass

    # pure-trajectory fallback (relaxed)
    x_final = float(fut12_ego_yfwd[-1, 0])
    if abs(x_final) > 1.8:
        return "LANE_LEFT" if x_final > 0 else "LANE_RIGHT"
    return None

# ======================================================================

# >>> small helper to safely toggle autopilot across CARLA versions
def _set_autopilot_safe(actor, enabled: bool):
    try:
        actor.set_autopilot(bool(enabled))
        return True
    except Exception:
        return False

# ---------- AGENTS image pruner (keeps .npz only) ----------
def start_agents_image_pruner(agents_dir: str, stop_event: Event, interval: float = 0.75):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".tif")
    def _loop():
        while not stop_event.is_set():
            if os.path.isdir(agents_dir):
                try:
                    for ext in exts:
                        for fp in glob.iglob(os.path.join(agents_dir, f"**/*{ext}"), recursive=True):
                            try:
                                os.remove(fp)
                            except Exception:
                                pass
                except Exception:
                    pass
            stop_event.wait(interval)
    t = Thread(target=_loop, daemon=True)
    t.start()
    return t

# ---------- Selection utility: categories + soft targets ----------
def bin_category(k: str) -> str:
    if k == "STOP": return "STOP"
    if k.startswith("STR_"): return "STRAIGHT"
    if k.startswith("LANE_"): return "LANE"
    return "TURN"

def choose_underrepresented_bin_weighted(counts: Dict[str,int], weights: Dict[str,float],
                                         avoid_keys: Optional[set] = None) -> str:
    avoid_keys = avoid_keys or set()
    items = []
    for k in BIN_KEYS:
        if k in avoid_keys: continue
        w = float(weights.get(k, 1.0))
        items.append((counts.get(k,0) * w, random.random(), k))
    if not items:
        items = [(counts.get(k,0), random.random(), k) for k in BIN_KEYS]
    items.sort(key=lambda t: (t[0], t[1]))
    return items[0][2]

def run_record_data(
    mq: multiprocessing.Queue,
    map_name: str,
    record_path: str,
    record_frames: int = 100,
    carla_port: int = 2000,
    num_vehicles: int = 100,
    num_oods: int = 200,
    weather: Optional[str] = None,
    lattice_pkl: Optional[str] = None,
    balance_state_path: Optional[str] = None,
    prune_agents_images: bool = True,
    # base bin-category weights (>=1 penalizes selection, <1 promotes)
    stop_weight: float = 1.4,
    straight_weight: float = 1.05,
    turn_weight: float = 1.0,
    lane_weight: float = 1.1,
    # strategy knobs
    avoid_stop_until_fill: bool = True,
    zero_bin_discount: float = 0.6,     # <1 boosts any bin currently at 0 effective count
    mild_turn_discount: float = 0.8,    # <1 boosts *_MILD
    # soft category targets (fractions roughly sum ~1)
    soft_target_stop: float = 0.08,
    soft_target_straight: float = 0.55,
    soft_target_turn: float = 0.27,
    soft_target_lane: float = 0.10,
    cat_alpha: float = 1.5,             # exponent for how strongly target deviations affect weights
    # STOP classification thresholds (tight)
    stop_len: float = 1.8,
    stop_yf: float = 1.4,
):
    """
    Collect frames in CARLA at 10Hz, grouped into balanced 6s micro-episodes.
    """
    global STOP_LEN_TIGHT, STOP_YF_TIGHT
    STOP_LEN_TIGHT = float(stop_len)
    STOP_YF_TIGHT  = float(stop_yf)

    stop_hb = Event()
    pruner_stop = Event()
    pruner_thread = None

    orig_settings = None
    tm = None
    last_tick = None
    drop_counts = {"pos": 0, "yaw": 0, "dt": 0}
    EP_HZ = 10.0
    DT = 1.0 / EP_HZ
    TICKS_PER_EP = 60  # 6s
    TOTAL_EP = max(1, record_frames // TICKS_PER_EP)

    # Balancing counters (per-run)
    bin_counts = _zeros_dict()
    lattice = load_lattice(lattice_pkl)
    mode_counts = {}

    # STOP metrics (visibility)
    stop_metrics = {
        "forced": 0,
        "detected_total": 0,
        "leak": 0,
        "oldrule_candidates": 0,
        "reassigned_to_STR_P0": 0,
    }

    # ---------- Persistent (across-run) balance state ----------
    persistent = {"per_map": {}, "global": _zeros_dict()}
    if balance_state_path:
        loaded = _load_json(balance_state_path, {})
        if isinstance(loaded, dict):
            persistent.update({k: loaded.get(k, v) for k, v in persistent.items()})
        for k in BIN_KEYS:
            persistent["global"][k] = int(persistent["global"].get(k, 0))
    per_map = persistent["per_map"]
    if map_name not in per_map or not isinstance(per_map[map_name], dict):
        per_map[map_name] = _zeros_dict()
    else:
        for k in BIN_KEYS:
            per_map[map_name][k] = int(per_map[map_name].get(k, 0))

    try:
        print("Saving to:", os.path.abspath(record_path))
        start_heartbeat(mq, stop_hb, interval=5.0)

        # ---- ENV + world ----
        env = CarlaEnvironment(carla_host='localhost', carla_port=carla_port, carla_timeout=30.0)
        world = env.world
        env.change_map(map_name)
        if weather is not None:
            env.set_weather(weather)

        # ---- SYNC 10Hz + TM ----
        orig_settings, tm = configure_sync(env, fps=int(EP_HZ), carla_port=carla_port, tm_port=None, seed=0)

        # ---- Spawn ego + traffic ----
        recorder = CarlaRecorder(env)
        agent_vehicle = env.spawn_agent_vehicle()
        env.spectator_track_vehicle(agent_vehicle)
        time.sleep(0.5)

        env.spawn_traffic(num_vehicles, 0)
        time.sleep(1.0)
        if num_oods > 0:
            env.spawn_oods(num_oods)
        time.sleep(2.0)

        # ---- Output dirs ----
        metadata_dir   = os.path.join(record_path, "metadata")
        raster_rgb_dir = os.path.join(record_path, "rasters_rgb")
        raster_lbl_dir = os.path.join(record_path, "rasters_lbl")
        sequences_dir  = os.path.join(record_path, "sequences")
        for p in (metadata_dir, raster_rgb_dir, raster_lbl_dir, sequences_dir):
            os.makedirs(p, exist_ok=True)

        # ---- Static map layers & rasterizer ----
        print("Building OpenDRIVE lanes…")
        static_layers = build_static_lane_polylines(world, sample_res_m=1.5)
        od_raster = OpenDriveRasterizer(static_layers, img_size=512, meters_ahead=50.0, meters_behind=20.0, meters_side=35.0)
        ppm, sz, a, b, s = od_raster.get_view_params()

        # Save OpenDRIVE text
        ox = ""
        try:
            ox = world.get_map().to_opendrive()
            with open(os.path.join(record_path, "map_opendrive.xml"), "w") as f:
                f.write(ox)
        except Exception:
            pass

        # Smoke-test raster
        try:
            tf0 = agent_vehicle.get_transform()
            ego_xy0 = (tf0.location.x, tf0.location.y)
            rgb0, lbl0 = od_raster.render(ego_xy0, tf0.rotation.yaw)
            rgb0.convert("RGB").save(os.path.join(record_path, "_debug_raster_rgb.png"))
            lbl0.save(os.path.join(record_path, "_debug_raster_lbl.png"))
        except Exception:
            traceback.print_exc()

        # ---- Begin record ----
        print("Record started.")
        recorder.start_record(save_path=record_path)

        # Start AGENTS image pruner (keeps .npz)
        if prune_agents_images:
            agents_dir = os.path.join(record_path, "agents")
            pruner_thread = start_agents_image_pruner(agents_dir, pruner_stop, interval=0.75)
            print("[PRUNER] Agents image pruner active (deleting images under 'agents/', keeping .npz).")

        # Pre-cache waypoints for episode placement
        wps_dense = find_waypoints(world, step=8.0)

        import carla
        ego_actor = getattr(agent_vehicle, "_vehicle", None) or getattr(agent_vehicle, "vehicle", None) \
                    or getattr(agent_vehicle, "_actor", None) or getattr(agent_vehicle, "actor", None)
        if ego_actor is None:
            raise RuntimeError("Could not resolve ego carla.Actor")

        # Ensure autopilot under TM control initially
        _set_autopilot_safe(ego_actor, True)

        saved_count = 0
        ep_id = 0

        # Soft target map
        SOFT_TARGETS = {
            "STOP": float(soft_target_stop),
            "STRAIGHT": float(soft_target_straight),
            "TURN": float(soft_target_turn),
            "LANE": float(soft_target_lane),
        }

        while saved_count < record_frames and ep_id < TOTAL_EP:
            # === Build effective counts (persistent + this run)
            effective = {k: per_map[map_name].get(k,0) + bin_counts.get(k,0) for k in BIN_KEYS}

            # category totals so far
            cat_counts = {"STOP":0, "STRAIGHT":0, "TURN":0, "LANE":0}
            total = 0
            for k,v in effective.items():
                cat_counts[bin_category(k)] += int(v)
                total += int(v)
            total = max(total, 1)

            # dynamic category pressure (ratio>1 => over target => penalize)
            dyn_cat_ratio = {}
            for cat, tgt in SOFT_TARGETS.items():
                desired = max(int(total * max(tgt, 1e-3)), 1)
                dyn_cat_ratio[cat] = (cat_counts[cat] + 1e-3) / (desired + 1e-3)

            # base weights per bin from category
            base_cat_w = {
                "STOP": float(stop_weight),
                "STRAIGHT": float(straight_weight),
                "TURN": float(turn_weight),
                "LANE": float(lane_weight),
            }

            # Avoid STOP early if requested and any movement bin is 0
            avoid = set()
            if avoid_stop_until_fill and any(effective[k]==0 for k in MOVEMENT_KEYS):
                avoid.add("STOP")

            # build dynamic per-bin weights
            dyn_w = {}
            for k in BIN_KEYS:
                cat = bin_category(k)
                w = base_cat_w[cat] * (dyn_cat_ratio[cat] ** float(cat_alpha))
                # boost never-seen bins
                if effective.get(k,0) == 0:
                    w *= float(zero_bin_discount)
                # gently favor mild turns
                if k.endswith("_MILD"):
                    w *= float(mild_turn_discount)
                dyn_w[k] = w

            # pick target
            target_bin = choose_underrepresented_bin_weighted(effective, dyn_w, avoid_keys=avoid)

            # === Place ego for this episode
            wp_chosen = None
            if target_bin == "STOP":
                wp_chosen = pick_straight_wp(wps_dense)
            elif target_bin.startswith("STR_"):
                if target_bin in ("STR_P0_0to8","STR_P1_8to18"):
                    wp_chosen = pick_straight_wp(wps_dense)
                else:
                    wp_chosen = random.choice(wps_dense) if wps_dense else None
            elif "TURN" in target_bin:
                # prefer gentle curves if asking for MILD
                if target_bin.endswith("_MILD"):
                    wp_chosen = pick_gentle_curve_wp(wps_dense)
                else:
                    wp_chosen = pick_junction_approach_wp(wps_dense)
            elif target_bin.startswith("LANE_"):
                side = "left" if target_bin=="LANE_LEFT" else "right"
                wp_chosen = pick_multilane_wp(wps_dense, side=side)

            if wp_chosen is None and wps_dense:
                wp_chosen = random.choice(wps_dense)

            if wp_chosen is not None:
                tf = wp_chosen.transform
                back = wp_chosen.previous(10.0)
                if back:
                    tf = back[0].transform
                try:
                    ego_actor.set_transform(tf)
                    world.tick()
                except Exception:
                    pass

            # Episode bias controller
            bias = EpisodeBias(target_bin)

            # Per-episode transient TM knobs
            try:
                tm.vehicle_min_speed(ego_actor, 0.0)
            except Exception:
                pass

            # STOP handling
            stop_mode = (target_bin == "STOP")
            restore_autopilot = False
            if stop_mode:
                if _set_autopilot_safe(ego_actor, False):
                    restore_autopilot = True
                try:
                    ego_actor.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True, steer=0.0))
                except Exception:
                    pass
                stop_metrics["forced"] += 1

            # Collect episode frames (and a local polyline for classification)
            ep_positions = []
            ep_yaws = []

            t0 = None
            ticks = 0
            while ticks < TICKS_PER_EP and saved_count < record_frames:
                snapshot = world.wait_for_tick()
                if snapshot is None:
                    continue
                t = snapshot.timestamp.elapsed_seconds
                if t0 is None: t0 = t
                t_ep = t - t0

                tf = ego_actor.get_transform()
                vel = ego_actor.get_velocity()
                ego_yaw = tf.rotation.yaw

                # sanity gate
                x = tf.location.x; y = tf.location.y
                if saved_count > 0 and last_tick is not None:
                    dt = t - last_tick["t"]
                    if dt <= 0.0 or dt > 0.25:
                        drop_counts["dt"] += 1
                        last_tick = {"x": x, "y": y, "yaw": ego_yaw, "t": t}
                        continue
                    scale = max(0.5, min(2.5, dt / DT))
                    step_lim = MAX_STEP_M_0P1S * scale
                    yaw_lim  = MAX_YAW_DEG_0P1S * scale
                    dx = math.hypot(x - last_tick["x"], y - last_tick["y"])
                    dyaw = _ang_diff_deg(ego_yaw, last_tick["yaw"])
                    if dx > step_lim:
                        drop_counts["pos"] += 1
                        last_tick = {"x": x, "y": y, "yaw": ego_yaw, "t": t}
                        continue
                    if dyaw > yaw_lim:
                        drop_counts["yaw"] += 1
                        last_tick = {"x": x, "y": y, "yaw": ego_yaw, "t": t}
                        continue
                last_tick = {"x": x, "y": y, "yaw": ego_yaw, "t": t}

                # apply episode bias
                try:
                    if stop_mode:
                        ego_actor.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True, steer=0.0))
                    else:
                        bias.apply_tick(ego_actor, tm, t_ep)
                except Exception:
                    pass

                # Raster
                ego_xy = (tf.location.x, tf.location.y)
                rgb, lbl = od_raster.render(ego_xy, ego_yaw)

                # Save frame
                saved_count += 1; ticks += 1
                rgb_path = os.path.join(raster_rgb_dir, f"frame_{saved_count:05d}.png")
                lbl_path = os.path.join(raster_lbl_dir,  f"frame_{saved_count:05d}.png")
                rgb.convert("RGB").save(rgb_path)
                lbl.save(lbl_path)

                # Minimal metadata
                ego_vel_list = [vel.x, vel.y, vel.z]
                ego_state = {
                    "timestamp": float(t),
                    "frame_number": int(snapshot.frame),
                    "id": int(ego_actor.id),
                    "position": [float(tf.location.x), float(tf.location.y), float(tf.location.z)],
                    "velocity": ego_vel_list + [speed_mag(ego_vel_list)],
                    "heading": float(ego_yaw),
                }
                frame_meta = {
                    "ego": ego_state,
                    "metadata": {
                        "map_name": map_name,
                        "weather": weather,
                        "bin_target": target_bin,
                        "episode_id": int(ep_id),
                        "episode_tick": int(ticks),
                    },
                    "raster_paths": {
                        "rgb": os.path.relpath(rgb_path, record_path),
                        "label": os.path.relpath(lbl_path, record_path)
                    },
                    "frame_index": int(saved_count)
                }
                with open(os.path.join(metadata_dir, f"frame_{saved_count:05d}.json"), "w") as f:
                    json.dump(frame_meta, f, indent=2)

                # Accumulate for local classification (relative to first pose)
                ep_positions.append([tf.location.x, tf.location.y])
                ep_yaws.append(ego_yaw)

                mq.put(saved_count)

            # === Restore state after STOP episode
            if stop_mode:
                try:
                    ego_actor.apply_control(carla.VehicleControl(throttle=0.0, brake=0.0, hand_brake=False))
                except Exception:
                    pass
                if restore_autopilot:
                    _set_autopilot_safe(ego_actor, True)

            # === Classify episode trajectory
            if len(ep_positions) >= 2:
                P = np.asarray(ep_positions, np.float32)
                x0,y0 = P[0]
                yaw0 = math.radians(ep_yaws[0])
                c,s = math.cos(-yaw0), math.sin(-yaw0)
                R = np.array([[c,-s],[s,c]], np.float32)
                Q = (P - np.array([x0,y0], np.float32)[None,:]) @ R.T
                yaw90 = np.array([[0,-1],[1,0]], np.float32)
                Qy = Q @ yaw90.T
                fut12 = resample_to_T(Qy[1:], 12)

                # Old vs tight STOP checks for metrics
                disp_y = float(fut12[-1,1])
                path_len = float(np.linalg.norm(np.diff(fut12, axis=0), axis=1).sum())
                old_stop = (path_len < OLD_STOP_LEN) or (disp_y < OLD_STOP_YF)
                if old_stop: stop_metrics["oldrule_candidates"] += 1

                # lane-change first (if any), else general classification
                lane_key = classify_lane_change(world, P, fut12)

                if target_bin == "STOP":
                    bin_key = "STOP"
                    stop_metrics["detected_total"] += 1
                else:
                    # Use tightened classification
                    base_key = classify_future_2hz(fut12)
                    if base_key == "STOP":
                        # Tight STOP hit while not targeting STOP -> leak but keep distribution healthy
                        stop_metrics["detected_total"] += 1
                        stop_metrics["leak"] += 1
                        bin_key = "STR_P0_0to8"
                        if old_stop:
                            stop_metrics["reassigned_to_STR_P0"] += 1
                    else:
                        bin_key = lane_key if lane_key is not None else base_key

                bin_counts[bin_key] = bin_counts.get(bin_key,0) + 1
                per_map[map_name][bin_key] = per_map[map_name].get(bin_key,0) + 1
                persistent["global"][bin_key] = persistent["global"].get(bin_key,0) + 1

                if lattice is not None:
                    try:
                        mi = nearest_mode_idx(lattice, fut12)
                        mode_counts[mi] = mode_counts.get(mi,0) + 1
                    except Exception:
                        pass

            ep_id += 1
            # lightly randomize after each episode to avoid getting stuck
            try:
                tm.vehicle_percentage_speed_difference(ego_actor, random.choice([-20,-30,-10,0]))
            except Exception:
                pass

        # ---- Stop recorder and signal done ----
        recorder.stop_record()
        mq.put('record finished')

        # ---- Save persistent state ----
        if balance_state_path:
            _save_json(balance_state_path, persistent)

        # ---- Final prune sweep (in case anything landed between loops) ----
        if prune_agents_images:
            try:
                agents_dir = os.path.join(record_path, "agents")
                if os.path.isdir(agents_dir):
                    for ext in (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".tif"):
                        for fp in glob.iglob(os.path.join(agents_dir, f"**/*{ext}"), recursive=True):
                            try: os.remove(fp)
                            except Exception: pass
            except Exception:
                pass

        # ---- Summaries ----
        print("\n=== Balanced bin counts (this run) ===")
        for k in BIN_KEYS:
            print(f"{k:>14}: {bin_counts.get(k,0)}")

        # category summary vs targets
        cat_tot = {"STOP":0, "STRAIGHT":0, "TURN":0, "LANE":0}
        for k,v in bin_counts.items():
            cat_tot[bin_category(k)] += int(v)
        total_run = max(sum(bin_counts.values()), 1)
        print("\n=== Category totals (run) vs soft targets ===")
        for cat in ["STRAIGHT","TURN","LANE","STOP"]:
            pct = 100.0 * cat_tot[cat] / total_run
            tgt = 100.0 * SOFT_TARGETS[cat]
            print(f"{cat:>9}: {cat_tot[cat]:4d}  ({pct:5.1f}% vs target {tgt:5.1f}%)")

        print("\n=== STOP metrics (visibility) ===")
        for k,v in stop_metrics.items():
            print(f"{k:>24}: {v}")

        print("\n=== CUMULATIVE bin totals (this map) ===")
        for k in BIN_KEYS:
            print(f"{k:>14}: {per_map[map_name].get(k,0)}")

        print("\n=== CUMULATIVE bin totals (GLOBAL across all maps) ===")
        for k in BIN_KEYS:
            print(f"{k:>14}: {persistent['global'].get(k,0)}")

        if mode_counts:
            print("\n=== Lattice nearest-mode counts (optional) ===")
            print("{", end="")
            first=True
            for k in sorted(mode_counts.keys()):
                if not first: print(", ", end="")
                first=False
                print(f"{k}: {mode_counts[k]}", end="")
            print("}")

        print(f"\n[SANITY] dropped frames (pos>{MAX_STEP_M_0P1S:.1f}m: {drop_counts['pos']}, "
              f"yaw>{MAX_YAW_DEG_0P1S:.1f}°: {drop_counts['yaw']}, dt bad: {drop_counts['dt']})")

    except Exception as e:
        print("Exception in run_record_data:", e)
        traceback.print_exc()
        mq.put('error')
    finally:
        # ---- ALWAYS restore sync & TM ----
        try:
            if tm: tm.set_synchronous_mode(False)
        except Exception: pass
        try:
            if 'env' in locals() and env and hasattr(env, "world"):
                env.world.apply_settings(orig_settings)
        except Exception: pass

        # stop pruner + heartbeat
        try:
            pruner_stop.set()
            if pruner_thread is not None: pruner_thread.join(timeout=2.0)
        except Exception: pass
        try:
            stop_hb.set()
        except Exception: pass
        try:
            if 'env' in locals(): env.close()
        except Exception: pass

def test_carla_connection(carla_port=2000, timeout=10):
    try:
        import carla
        client = carla.Client('localhost', carla_port); client.set_timeout(timeout)
        _ = client.get_world()
        print(f"Successfully connected to CARLA on port {carla_port}")
        return True
    except Exception as e:
        print(f"Failed to connect to CARLA on port {carla_port}: {e}")
        return False

def collect_data(map_name, record_name, record_frames=600, carla_port=2000,
                 num_vehicles=80, num_oods=120, weather=None, lattice_pkl=None,
                 balance_state_path=None, prune_agents_images=True,
                 stop_weight=1.4, straight_weight=1.05, turn_weight=1.0, lane_weight=1.1,
                 avoid_stop_until_fill=True, zero_bin_discount=0.6, mild_turn_discount=0.8,
                 soft_target_stop=0.08, soft_target_straight=0.55, soft_target_turn=0.27, soft_target_lane=0.10,
                 cat_alpha=1.5, stop_len=1.8, stop_yf=1.4):
    record_path = os.path.join("data", record_name)
    os.makedirs(record_path, exist_ok=True)

    print('Testing connection to CARLA Docker instance...')
    if not test_carla_connection(carla_port):
        print("Cannot connect to CARLA. Make sure CARLA Docker container is running.")
        return False

    success=False
    recorder_mq=multiprocessing.Queue()
    worker_timeout=datetime.timedelta(minutes=60)

    try:
        last_msg=datetime.datetime.now()
        print('Starting record process...')
        proc=multiprocessing.Process(
            target=run_record_data,
            args=[recorder_mq, map_name, record_path],
            kwargs=dict(record_frames=record_frames, carla_port=carla_port,
                        num_vehicles=num_vehicles, num_oods=num_oods,
                        weather=weather, lattice_pkl=lattice_pkl,
                        balance_state_path=balance_state_path,
                        prune_agents_images=prune_agents_images,
                        stop_weight=stop_weight, straight_weight=straight_weight,
                        turn_weight=turn_weight, lane_weight=lane_weight,
                        avoid_stop_until_fill=avoid_stop_until_fill,
                        zero_bin_discount=zero_bin_discount, mild_turn_discount=mild_turn_discount,
                        soft_target_stop=soft_target_stop, soft_target_straight=soft_target_straight,
                        soft_target_turn=soft_target_turn, soft_target_lane=soft_target_lane,
                        cat_alpha=cat_alpha, stop_len=stop_len, stop_yf=stop_yf),
        )
        proc.start()
        while proc.is_alive():
            try:
                msg=recorder_mq.get_nowait()
                last_msg=datetime.datetime.now()
                if not (isinstance(msg, tuple) and msg and msg[0]=='hb'):
                    print(f"Recording progress: {msg}")
                if msg=='record finished':
                    success=True; break
                elif msg=='error':
                    print("Error occurred during recording"); break
            except queue.Empty:
                pass
            if datetime.datetime.now()-last_msg > worker_timeout:
                print('Recording timeout, stopping process.'); break
            time.sleep(0.25)
    except KeyboardInterrupt:
        print('Keyboard interrupt received, stopping data collection')
    except Exception as e:
        print(f"Error during data collection: {e}")
        traceback.print_exc()
    finally:
        try:
            if proc.is_alive():
                proc.terminate(); proc.join(10)
                if proc.is_alive(): proc.kill()
        except Exception: pass

    if success:
        print(f'Data collection success for {map_name}, data recorded to {record_path}.')
    else:
        print(f'Data collection failed. Removing collected partial data.')
        if os.path.exists(record_path): shutil.rmtree(record_path)
    return success

def main():
    parser=argparse.ArgumentParser(description='Collect balanced micro-episodes from CARLA')
    parser.add_argument('map_name', type=str)
    parser.add_argument('record_name', type=str)
    parser.add_argument('-f','--frames', type=int, default=600, help="total frames to save (10Hz)")
    parser.add_argument('-p','--carla_port', type=int, default=2000)
    parser.add_argument('-v','--num_vehicles', type=int, default=80)
    parser.add_argument('-o','--num_oods', type=int, default=120)
    parser.add_argument('-w','--weather', type=str, choices=WEATHER_PRESETS)
    parser.add_argument('--lattice_pkl', type=str, default=os.environ.get("LATTICE_PKL", ""), help="optional: path to fixed lattice .pkl for reporting coverage")
    parser.add_argument('--balance_state', type=str, default=os.path.join("data","_balance_state.json"),
                        help="path to JSON file to persist bin counts across runs")
    parser.add_argument('--keep_agents_images', action='store_true',
                        help="If set, do NOT prune image files under data/<record_name>/agents/. Default prunes images.")
    # base weights
    parser.add_argument('--stop_weight', type=float, default=1.4)
    parser.add_argument('--straight_weight', type=float, default=1.05)
    parser.add_argument('--turn_weight', type=float, default=1.0)
    parser.add_argument('--lane_weight', type=float, default=1.1)
    # strategy / soft targets
    parser.add_argument('--allow_stop_early', action='store_true', help="Allow selecting STOP even if some non-STOP bins have zero count")
    parser.add_argument('--zero_bin_discount', type=float, default=0.6)
    parser.add_argument('--mild_turn_discount', type=float, default=0.8)
    parser.add_argument('--soft_target_stop', type=float, default=0.08)
    parser.add_argument('--soft_target_straight', type=float, default=0.55)
    parser.add_argument('--soft_target_turn', type=float, default=0.27)
    parser.add_argument('--soft_target_lane', type=float, default=0.10)
    parser.add_argument('--cat_alpha', type=float, default=1.5)
    # STOP thresholds
    parser.add_argument('--stop_len', type=float, default=1.8)
    parser.add_argument('--stop_yf', type=float, default=1.4)
    args=parser.parse_args()

    ok = collect_data(map_name=args.map_name, record_name=args.record_name,
                      record_frames=args.frames, carla_port=args.carla_port,
                      num_vehicles=args.num_vehicles, num_oods=args.num_oods,
                      weather=args.weather, lattice_pkl=(args.lattice_pkl if args.lattice_pkl else None),
                      balance_state_path=args.balance_state,
                      prune_agents_images=(not args.keep_agents_images),
                      stop_weight=args.stop_weight, straight_weight=args.straight_weight,
                      turn_weight=args.turn_weight, lane_weight=args.lane_weight,
                      avoid_stop_until_fill=(not args.allow_stop_early),
                      zero_bin_discount=args.zero_bin_discount,
                      mild_turn_discount=args.mild_turn_discount,
                      soft_target_stop=args.soft_target_stop,
                      soft_target_straight=args.soft_target_straight,
                      soft_target_turn=args.soft_target_turn,
                      soft_target_lane=args.soft_target_lane,
                      cat_alpha=args.cat_alpha, stop_len=args.stop_len, stop_yf=args.stop_yf)
    if not ok: exit(1)

if __name__ == '__main__':
    # main()  # enable for single-run
    # Example batch (tuned for your target mix):
    for map_name in ['Town02']:
        for idx in range(0, 16):
            print(f'Collecting balanced data for {map_name}_{idx}')
            weather=random.choice(WEATHER_PRESETS)
            ok=collect_data(map_name=map_name, record_name=f'{map_name}_{idx}',
                            record_frames=200, carla_port=2000,
                            num_vehicles=100, num_oods=0, weather=weather,
                            lattice_pkl="/data/home/dal667613/NEW_extracted_data/data/lattices/epsilon_8.pkl",
                            balance_state_path=os.path.join("data","_balance_state.json"),
                            prune_agents_images=True,
                            stop_weight=1.4, straight_weight=1.05, turn_weight=1.0, lane_weight=1.1,
                            avoid_stop_until_fill=True,    # hold STOP until movement bins seen
                            zero_bin_discount=0.6,         # prefer never-seen bins
                            mild_turn_discount=0.8,        # favor *_MILD a bit
                            soft_target_stop=0.10,         # ~10%
                            soft_target_straight=0.52,     # ~52%
                            soft_target_turn=0.28,         # ~28%
                            soft_target_lane=0.10,         # ~10%
                            cat_alpha=1.5, stop_len=1.8, stop_yf=1.4)
            if not ok:
                print(f"Failed to collect data for {map_name}_{idx}, continuing...")
            time.sleep(2)


Testing connection to CARLA Docker instance...
Failed to connect to CARLA on port 2000: time-out of 10000ms while waiting for the simulator, make sure the simulator is ready and connected to localhost:2000
Cannot connect to CARLA. Make sure CARLA Docker container is running.
Failed to collect data for Town02_5, continuing...
Testing connection to CARLA Docker instance...


KeyboardInterrupt: 